# 00 - Data Preparation

Loads the raw Healthy Minds Study 2024-25 data, builds derived variables 
(PHQ-9/GAD-7 totals, screening flags, demographics, belonging, stigma), 
and filters to emerging adults (18-24).

**Input:** `data/HMS_2024-25.csv`  
**Output:** `data/emerging_adults_clean.csv`

In [1]:
import pandas as pd
import numpy as np

# Load the raw Healthy Minds Study 2024-25 data
df = pd.read_csv("../data/HMS_2024-2025_PUBLIC_instchars.csv", low_memory=False)

print(df.shape)

(84735, 1608)


## Build PHQ-9 and GAD-7 totals

The Healthy Minds data stores PHQ-9 and GAD-7 responses on a 1-4 scale, but the 
standard clinical scoring uses 0-3. Subtract 1 from each item before summing so 
the totals match published thresholds (PHQ-9: 0-27, GAD-7: 0-21). `min_count` 
requires all items to be answered, so partial responses become missing rather 
than artificially low scores.

In [2]:
# build composite scores
phq9_items = [f"phq9_{i}" for i in range(1, 10)]
gad7_items = [f"gad7_{i}" for i in range(1, 8)]

# subtract 1 from each item to convert 1-4 coding back to 0-3 (official scoring)
df["phq9_total"] = (df[phq9_items] - 1).sum(axis=1, min_count=9)
df["gad7_total"] = (df[gad7_items] - 1).sum(axis=1, min_count=7)

## Create clinical screening flags

Flag students at or above the clinical cutoff (≥10) for depression and anxiety. 
Students with missing totals stay missing rather than counted as negative.

In [3]:
# screening flag with 10 as standard clinical cutoff
df["depression_screen"] = (df["phq9_total"] >= 10).astype(int)
df["anxiety_screen"] = (df["gad7_total"] >= 10).astype(int)
# drop NaNs
df.loc[df["phq9_total"].isna(), "depression_screen"] = pd.NA
df.loc[df["gad7_total"].isna(), "anxiety_screen"] = pd.NA

## Build gender variable

The survey records gender identity as a set of separate yes/no indicator columns 
(one each for woman, man, nonbinary, trans, genderqueer, self-described) rather 
than a single variable, and respondents can mark more than one. To analyze how 
mental health and loneliness vary by gender, I collapse these indicators into one 
categorical variable.

Because the categories are not mutually exclusive, students who marked multiple 
identities are assigned using a priority order that lists gender minority 
identities first (e.g., a student marking both "trans" and "woman" is classified 
as Trans).

In [4]:
# gender condition
conditions = [
    df["gender_nonbin"] == 1,
    df["gender_trans"] == 1,
    df["gender_queer"] == 1,
    df["gender_female"] == 1,
    df["gender_male"] == 1,
    df["gender_selfID"] == 1,
]
choices = ["Nonbinary", "Trans", "Genderqueer", "Woman", "Man", "Self-described"]

df["gender"] = np.select(conditions, choices, default=None)

## Select analysis columns and filter to emerging adults

Keep only the variables used in the analysis, then restrict to emerging adults 
(ages 18-24). Row counts before and after the filter are printed as a diagnostic.

In [5]:
# pull relevant columns
working_cols = [
    # identifiers and demographics
    "responseid", "age", "gender",
    # mental health scores and screens
    "phq9_total", "gad7_total", "depression_screen", "anxiety_screen",
    # loneliness items
    "lonesc", "lonely",
    "lone_lackcompanion", "lone_leftout", "lone_isolated",
    # stress items
    "FinStress", "stress1", "stress2", "stress3", "stress4",
    # sexual orientation items
    "sexual_h", "sexual_l", "sexual_g", "sexual_bi", "sexual_queer",
    "sexual_quest", "sexual_pan", "sexual_asexual", "sexual_selfID", 
    # race items
    "race_black", "race_ainaan", "race_asian", "race_his", 
    "race_pi", "race_mides", "race_white", "race_other", 
    # belonging items
    "belong1", "belong2", "belong8", "belong9",
    # stigma items
    "stig_per_1", "stig_per_2", "stig_per_3",
    "stig_pcv_1", "stig_pcv_2", "stig_pcv_3",
    "stig_self_1", "stig_self_2", "stig_self_3"]

my_cols = df[working_cols].copy()

print("Working dataset:", my_cols.shape)

# filter to emerging adults (18-24)
emerging_adults = my_cols[my_cols["age"].between(18, 24)].copy()
print("Emerging adults (18-24):", len(emerging_adults))

Working dataset: (84735, 47)
Emerging adults (18-24): 61288


## Build sexual orientation variable

Like gender, sexual orientation is stored as separate indicator columns. I collapse 
them into one categorical variable using priority order, so students marking multiple 
identities are assigned a single label (minority identities take precedence over 
heterosexual).

In [6]:
conditions = [
    emerging_adults["sexual_bi"] == 1,
    emerging_adults["sexual_pan"] == 1,
    emerging_adults["sexual_l"] == 1,
    emerging_adults["sexual_g"] == 1,
    emerging_adults["sexual_queer"] == 1,
    emerging_adults["sexual_asexual"] == 1,
    emerging_adults["sexual_quest"] == 1,
    emerging_adults["sexual_selfID"] == 1,
    emerging_adults["sexual_h"] == 1,
]
choices = ["Bisexual", "Pansexual", "Lesbian", "Gay", "Queer",
           "Asexual", "Questioning", "Self-described", "Heterosexual"]
emerging_adults["sexual_orientation"] = np.select(conditions, choices, default=None)
print(emerging_adults["sexual_orientation"].value_counts(dropna=False))

sexual_orientation
Heterosexual      37593
Bisexual           9446
None               3529
Lesbian            2283
Queer              2165
Pansexual          1780
Asexual            1273
Gay                1247
Questioning        1240
Self-described      732
Name: count, dtype: int64


## Build race/ethnicity status

Collapse the separate race indicator columns into a binary White vs. Racial minority 
variable. Students marking any non-white identity (including multiracial) are 
classified as a racial minority.

In [7]:
# classify racial minority status
non_white_cols = ["race_black", "race_ainaan", "race_asian", 
                  "race_his", "race_pi", "race_mides", "race_other"]

marked_non_white = (emerging_adults[non_white_cols] == 1).any(axis=1)
marked_white = emerging_adults["race_white"] == 1

emerging_adults["race_status"] = np.select(
    [marked_non_white, marked_white & ~marked_non_white],
    ["Racial minority", "White"],
    default=None
)

print(emerging_adults["race_status"].value_counts(dropna=False))

race_status
White              30381
Racial minority    29891
None                1016
Name: count, dtype: int64


## Build campus belonging composite

Belonging items are on a 1-6 scale. Two items (belong1, belong2) are worded in the 
opposite direction, so they are reverse-coded (7 - value) before summing, so that 
higher scores mean more belonging.

In [8]:
for item in ["belong1", "belong2"]:
    emerging_adults[f"{item}_r"] = 7 - emerging_adults[item]

belong_corrected = ["belong1_r", "belong2_r", "belong8", "belong9"]
emerging_adults["belong_total"] = emerging_adults[belong_corrected].sum(axis=1, min_count=4)
print(emerging_adults["belong_total"].describe().round(2))

count    24175.00
mean        15.36
std          3.99
min          4.00
25%         13.00
50%         15.00
75%         18.00
max         24.00
Name: belong_total, dtype: float64


## Build stigma composites

Three stigma scales (personal, perceived, self). Items are reverse-coded where 
needed so that higher scores consistently mean more stigma. One perceived-stigma 
item (stig_pcv_2) is already worded so high = high stigma and is left as-is.

In [9]:
reverse_code_stigma = ["stig_per_1", "stig_per_2", "stig_per_3", "stig_pcv_1", "stig_pcv_3"]
for item in reverse_code_stigma:
    emerging_adults[f"{item}_r"] = 7 - emerging_adults[item]

personal_items = ["stig_per_1_r", "stig_per_2_r", "stig_per_3_r"]
perceived_items = ["stig_pcv_1_r", "stig_pcv_2", "stig_pcv_3_r"]
self_items = ["stig_self_1", "stig_self_2", "stig_self_3"]

emerging_adults["stigma_personal"] = emerging_adults[personal_items].sum(axis=1, min_count=3)
emerging_adults["stigma_perceived"] = emerging_adults[perceived_items].sum(axis=1, min_count=3)
emerging_adults["stigma_self"] = emerging_adults[self_items].sum(axis=1, min_count=3)

print(emerging_adults[["stigma_personal", "stigma_perceived", "stigma_self"]].describe().round(2))

       stigma_personal  stigma_perceived  stigma_self
count         26385.00          26372.00     24782.00
mean              8.88              8.04         6.42
std               1.84              2.90         2.87
min               3.00              3.00         3.00
25%               8.00              6.00         4.00
50%               8.00              8.00         6.00
75%               9.00             10.00         8.00
max              18.00             18.00        15.00


## Save cleaned dataset

Write the cleaned analysis sample to `data/`

In [10]:
emerging_adults.to_csv("../data/emerging_adults_clean.csv", index=False)